# What changed in the room? — Colab runner

This notebook runs the full pipeline end-to-end against a project folder you provide. It installs dependencies, aligns the room frame (stage 2), finds change clusters with a visibility-aware voxel diff (stage 3), runs Claude Haiku 4.5 on each flagged region (stage 4), and shows you the final report, the interactive 3D view, and seven walkthrough slides inline.

**What you need**

- A folder containing the project source files and `data/recon_da3.npz` (the shared 3D reconstruction from stage 1 — the sample run uses the included one).
- An Anthropic API key for stage 4 (free Haiku 4.5 trial credit is enough).
- No GPU required. If one is available (*Runtime → Change runtime type → GPU*), the pipeline uses it automatically.

Run the cells top to bottom.

## 1. Install dependencies

In [ ]:
!pip install -q torch transformers anthropic plotly scipy pillow matplotlib

## 2. Point at your project folder

Pick **one** of the three options below and uncomment it. Each sets your working directory to the project folder (which should contain `recon.py`, `vlm_reason.py`, `data/recon_da3.npz`, etc.).

In [ ]:
# Option A — clone from GitHub (fastest if the repo is public)
# !git clone -q https://github.com/YOUR_USER/scene-change
# %cd scene-change

# Option B — upload a zip of the project folder
# from google.colab import files
# uploaded = files.upload()        # pick your scene-change.zip
# !unzip -q scene-change.zip -d .
# %cd scene-change

# Option C — mount Google Drive (recommended for repeated runs)
from google.colab import drive
drive.mount('/content/drive')
PROJECT_PATH = '/content/drive/MyDrive/scene-change'  # edit if your folder lives elsewhere
%cd {PROJECT_PATH}

In [ ]:
import os
assert os.path.exists('data/recon_da3.npz'), (
    'data/recon_da3.npz not found. Generate it with notebooks/stage1_reconstruct_da3_kaggle.ipynb '
    'on Kaggle (needs ~16 GB VRAM), or copy the included sample into your project folder.'
)
print('Project folder OK.')
!ls -1 *.py && echo --- && ls -1 data/

## 3. Stage 2 — align the room frame

Fits the floor plane, snaps the walls to the x/y axes, and places the origin at the floor corner so all coordinates are positive and `z` is up.

Writes `data/room_frame.npz`.

In [ ]:
!python room_frame.py

## 4. Stage 3 — visibility-aware voxel diff

Voxelises both sessions at 3 cm, flags candidate change voxels (occupied in one session, empty in the other), and runs a per-voxel visibility test against every frame of the opposite session. Only confirmed changes become clusters.

Writes `data/changes.json`, `data/changes_voxels.npz`, and (via `show_clusters.py`) evidence sheets to `out/clusters_*.png`.

In [ ]:
!python diff.py

In [ ]:
!python show_clusters.py

import os
from IPython.display import Image, display
for p in sorted(p for p in os.listdir('out') if p.startswith('clusters_')):
    display(Image(filename=f'out/{p}'))

## 5. Stage 4 — VLM semantics

For each flagged cluster, projects its voxels into the owning session's best frame, uses them as a SAM point prompt to get a clean object crop, and asks Claude Haiku 4.5 to describe and classify it. Candidate `(removed, added)` pairs are then compared pairwise. The final step composes a short, human-readable report.

Writes `data/vlm.json`, `out/vlm_crops/`, and `out/report.md`.

In [ ]:
# Securely enter your Anthropic API key (not stored on disk, not echoed back).
import getpass, os
os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Anthropic API key: ')

In [ ]:
!python vlm_reason.py

## 6. Results

The Markdown report Haiku composed, plus the interactive 3D view.

In [ ]:
from IPython.display import Markdown
Markdown(open('out/report.md').read())

In [ ]:
!python view3d_vlm.py
from IPython.display import HTML
HTML(open('out/changes_3d_vlm.html').read())

## 7. Walkthrough slides

Seven PNGs tracing one example from input frames to final findings. Useful for building a video or presentation.

In [ ]:
!python make_showcase.py

import os
from IPython.display import Image, display
for p in sorted(os.listdir('showcase')):
    if p.endswith('.png'):
        print(p)
        display(Image(filename=f'showcase/{p}'))